In [ ]:
from mpi4py import MPI
import coqui

# Create CoQui MPI handler and set logging verbosity in the beginning
coqui_mpi = coqui.MpiHandler()
coqui.set_verbosity(coqui_mpi, output_level=1)

# GW Approximation

<figure style="text-align: center;">
 <img src="../../images/gw_pentagon.png" alt="GW pentagon" width="40%">
 <figcaption><em>Figure 1:</em> GW self-consistent equations.</figcaption>
</figure>

In this notebook, we will focus on the GW approximation, a widely used many-body perturbation theory for describing excited-state properties of weakly correlated materials. 

[A brief description on the physics of the GW approximation and its applications in materials science.]

As show in Figure 1, the GW approximation corresponds to the lowest-order approximation to the vertex function $\Gamma^{GW}=1$ in Hedin's equations, so that the irreducible polarizability is approximated as $\chi^{GW}=-GG$ and the self-energy is given by $\Sigma^{GW}=GW$.

[A brief introduction to GW equations in general space-time basis, the numerical representation adapted in CoQui (i.e. discrete single-particle basis) will be covered in the following sections.]

The resulting script also serves as a reusable template for other electronic-structure solvers in CoQuí.

**What you will learn**

1. How to construct a THC Coulomb Hamiltonian for many-body electronic structure calculations.

2. Running GW method with different levels of self-consistency

3. Visualize GW band structure and spectral function.

> Note: Numerical parameters in this notebook are intentionally small for interactive runtime. They are not production settings; perform convergence studies for quantitative results.

First, copy the pre-computed inputs required for this notebook.

This prepares local copies of:

- `out_222/` (QE outputs for Section 1 and 2)

- `out_777/` (QE outputs for Section 3)

- `mlwf_777/si.mlwf.h5` (Wannier file for interpolation)

- `coqui_777/si.mbpt.h5` and `coqui_777/si_qpg0w0.mbpt.h5` (precomputed GW checkpoints)


In [ ]:
%%bash
python copy_notebook_inputs.py

### Section 1: Build the Coulomb Hamiltonian for GW

<figure style="text-align: center;">
 <img src="../../images/coqui_problem_setup.png" alt="Problem setup in CoQuí" width="60%">
 <figcaption><em>Figure 1:</em> Problem setup for a many-body simulation.</figcaption>
</figure>

The many-body calculations in CoQui rely on the second quantized Hamiltonian, which consists of a non-interacting part and a Coulomb interaction part. The non-interacting Hamiltonian, as we have seen in the previous notebook, is defined by the `Mf` object. However, to perform many-body calculations such as GW, we also need to construct the Coulomb interaction Hamiltonian, which captures the electron-electron interactions in the system.

In this section, we will be working on construction of the Coulomb Hamiltonian: 
$$
\hat{H}_{\mathrm{int}} = \frac{1}{2}\sum _{ijkl}V^{\textbf{k}_1\textbf{k}_2\textbf{k}_3\textbf{k}_4} _{ijkl} c^{\textbf{k}_1\dagger} _{i} c^{\textbf{k}_3\dagger} _{k} c^{\textbf{k}_4} _{l} c^{\textbf{k}_2} _{j}
$$
where the Coulomb matrix elements are defined as:
$$
V^{\textbf{k}_1\textbf{k}_2\textbf{k}_3\textbf{k}_4} _{ijkl} = \int d\textbf{r} \int d\textbf{r}' \phi^{\textbf{k}_1*} _{i} (\textbf{r})\phi^{\textbf{k}_2} _{j}(\textbf{r})\frac{1}{|\textbf{r}-\textbf{r}'|}\phi^{\textbf{k}_3*} _{k}(\textbf{r}')\phi^{\textbf{k}_4} _{l}(\textbf{r}')
$$
This is the most general form of Coulomb Hamiltonian, capturing both *local* and *non-local* interactions among *all* electronic degrees of freedom. Here, the momentum transferred between the product basis 
$\rho^{\textbf{k}_{1}\textbf{k}_{2}}_{ij}(\textbf{r}) = \phi^{\textbf{k}_{1}*}_{i}(\textbf{r})\phi^{\textbf{k}_{2}}_{j}(\textbf{r})$
(also known as the pair densities) satisfies momentum conservation: 

$$
\textbf{k}_1 - \textbf{k}_2 + \textbf{G} = \textbf{k}_3 - \textbf{k}_4,
$$

where $\textbf{G}$ represents a reciprocal lattice vector of the system.

#### 🔹 THC Coulomb construction

In CoQui, the Coulomb interaction is handled by the `ThcCoulomb` class, in which the Coulomb matrix elements are represented in the tensor hypercontraction (THC) representation, thanks to the low-rank structure of the Coulomb tensor. 

$$
V^{\mathbf{k}_1\mathbf{k}_2\mathbf{k}_3\mathbf{k}_4}_{ijkl}
\approx \sum_{\mu\nu}^{N_\mu} X^{\mathbf{k}_1*}_{\mu i} X^{\mathbf{k}_2}_{\mu j} V^{\mathbf{q}}_{\mu\nu} X^{\mathbf{k}_3*}_{\nu k} X^{\mathbf{k}_4}_{\nu l}.
$$

where $\textbf{q} = \textbf{k}_{1} - \textbf{k}_{2}$ and the greek letters $\mu$ and $\nu$ label the THC auxiliary basis functions. 

A `ThcCoulomb` obejct is constructed through:

> ```python
> coqui.make_thc_coulomb(mf: Mf, params: dict) -> ThcCoulomb
> ```

Key parameters used here:
- `thresh` (default: 1e-5): THC threshold controlling the size of the THC auxiliary basis.

- `ecut` (default: 0.4 * Mf.ecutrho()): plane-wave cutoff (Hartree).

- `save` (default: ""): HDF5 file to save the THC Coulomb matrix elements ("" means no saving). 

> Tip: The resulting `ThcCoulomb` object can be constructed once and reused for multiple many-body calculations.

> Tip: For complete parameter documentation, use `help(coqui.make_thc_coulomb)` in Python.

#### 🔹 Example
In the following example, we build `Mf` from QE outputs and then construct a THC Coulomb Hamiltonian for silicon.

In [ ]:
# Step 1: Build mean-field object
mf_params = {
    "prefix": "si",
    "outdir": "out_222",
    "nbnd": 20,
}
mf = coqui.make_mf(coqui_mpi, params=mf_params, mf_type="qe")

# Step 2: Build THC Coulomb Hamiltonian
thc_params = {
    "thresh": 1e-3,
}
thc = coqui.make_thc_coulomb(mf=mf, params=thc_params)

Reflection questions:
1. What FFT mesh and number of plane waves are reported?

2. How many THC auxiliary basis (i.e., interpolation points) are used in the THC compression?


#### 🔹 Hands-on — Save and reload the THC Hamiltonian

Goal: caching the Coulomb Hamiltonian to disk to avoid recomputation.

Steps:
1. Add a `save` h5 file name to `thc_params` (e.g., `"save": "thc.coulomb.h5"`).

2. Build `ThcCoulomb` and confirm data are written to HDF5.

3. Rebuild with the same parameters and confirm it is loaded from file, instead of repeating the full construction.


In [ ]:
# Step 1: Save the THC Hamiltonian
thc_params["save"] = "thc.coulomb.h5"
thc = coqui.make_thc_coulomb(mf=mf, params=thc_params)

# Step 2: Reload from the same file
thc_reloaded = coqui.make_thc_coulomb(mf=mf, params=thc_params)

### Section 2: GW approximation

<figure style="text-align: center;">
 <img src="../../images/coqui_simulation_gw.png" alt="GW simulation in CoQuí" width="60%">
 <figcaption><em>Figure 2:</em> GW Dyson-SCF simulation stage.</figcaption>
</figure>

With a full second quantized Hamiltonian in hand, we are ready to run a GW simulation. 

As shown in Figure 2, internally in CoQui, the workloads of solving Hedin's equations are divided into several independent, self-contained modules, each responsible for a specific task such as computing the screened Coulomb interaction, evaluating the self-energy, and performing self-consistency loops. This modular design allows for flexibility in different self-consistency approaches and adding higher-order diagrams beyond GW (as we will see in the GW+EDMFT tutorials), while maintaining a similar computational workflow and input structure for users.

The flexibility of CoQui's GW implementation allows users to easily control the level of self-consistency while maintain similar computational workflow and input. In the following, we will show how to run both the GW Dyson-SCF and QP-SCF simulations with CoQui.

<figure style="text-align: center;">
 <img src="../../images/gw_pentagon_with_coqui_module.png" alt="GW simulation in CoQuí" width="40%">
 <figcaption><em>Figure 2:</em> Workload division of GW simulation in CoQuí.</figcaption>
</figure>

[Elaborate on the unique features of CoQui's GW implementation]

Relying on the efficient Fourtier transform on the imaginary axis as well as the THC representation of the Coulomb interaction, the GW method is implemented in CoQui with a favorable cubic scaling with respect to system size and linear scaling with repsect to the number of k-points.



#### 🔹 Quasiparticle GW approximation

This section runs one GW QP-SCF step and inspects the imaginary-axis setup.

> ```python
> coqui.run_evgw(params: dict, h_int: coqui.ThcCoulomb)
> ```

Function inputs:
- `params`: dictionary of GW parameters.

- `h_int`: Coulomb Hamiltonian in THC format (from Section 1).

Key parameters in `params`:

- `outdir` (default: `./`): output directory.

- `prefix` (required): prefix (`{output}.mbpt.h5`).

- `niter`(default: 1): Number of self-consistent iterations.

- `beta` (default: 1000): inverse temperature ($a.u.^{-1}$).

- `iaft_prec` (default: `"medium"`): precision level (`"low"`, `"medium"`, `"high"`).

For complete parameter options, use `help(coqui.run_evgw)`.

Cost and memory scale approximately linearly with the number of imaginary-axis basis functions.

#### 🔹 Example

In the following example, we run a single GW iteration with moderate imaginary-axis settings.

In [ ]:
# Step 1: Build mean-field object
mf_params = {
    "prefix": "si",
    "outdir": "out_222",
    "nbnd": 20,
}
mf = coqui.make_mf(coqui_mpi, params=mf_params, mf_type="qe")

# Step 2: Build THC Coulomb Hamiltonian
thc_params = {
    "thresh": 1e-3,
}
thc = coqui.make_thc_coulomb(mf=mf, params=thc_params)

gw_params = {
    "outdir": "./",
    "output": "si.evgw",
    "niter": 1,
    "beta": 100,
    "iaft": {
        "prec": "medium"
    },
}
coqui.run_evgw(params=gw_params, h_int=thc)

#### 🔹 Hands-on — GW band structure 

Goal: compute and visualize the GW band structure.


#### 🔹 Full-frequency GW approximation

This section runs one GW Dyson-SCF step and inspects the imaginary-axis setup.

> ```python
> coqui.run_gw(params: dict, h_int: coqui.ThcCoulomb)
> ```

Function inputs:
- `params`: dictionary of GW parameters.

- `h_int`: Coulomb Hamiltonian in THC format (from Section 1).

Key parameters in `params`:

- `outdir` (default: `./`): output directory.

- `prefix` (required): output prefix (`{output}.mbpt.h5`).

- `niter` (default: 1): Dyson-SCF iterations.

- `beta` (default: 1000): inverse temperature ($a.u.^{-1}$).

- `iaft_prec` (default: `"medium"`): precision level (`"low"`, `"medium"`, `"high"`).

For complete parameter options, use `help(coqui.run_gw)`.

Cost and memory scale approximately linearly with the number of imaginary-axis basis functions.

#### 🔹 Example

In the following example, we run a single GW iteration with moderate imaginary-axis settings.

In [ ]:
gw_params = {
    "restart": False,
    "output": "gw",
    "niter": 1,
    "beta": 500,
    "wmax": 1.8,
    "iaft_prec": "medium",
}
coqui.run_gw(params=gw_params, h_int=thc)

#### 🔹 Hands-on — Explore GW imaginary-axis settings

Use one scaffolded workflow to compare how `beta`, `wmax`, and `iaft_prec` affect the GW log and results.

Steps:

1. Run the baseline case (`beta=500`, `wmax=1.8`, `iaft_prec="medium"`).

2. Increase `beta` (e.g., `5000`) and compare sampling sizes.

3. Decrease `wmax` (e.g., `0.18`) and check for leakage warnings.

4. Change `iaft_prec` (`"low"` / `"high"`) and compare energy differences.

Checklist:

- You can map major log blocks to HF, screened interaction, and GW self-energy stages.

- You identify how many samplings are built for fermionic and bosonic meshes.

- You confirm `{output}.mbpt.h5` is written and compare energies across runs.


In [ ]:
# Edit this list to run and compare different IAFT/GW settings.

cases = [
    {"name": "baseline", "beta": 500, "wmax": 1.8, "iaft_prec": "medium"},
    {"name": "low_temperature", "beta": 5000, "wmax": 1.8, "iaft_prec": "medium"},
    {"name": "small_wmax", "beta": 500, "wmax": 0.18, "iaft_prec": "medium"},
    {"name": "small_wmax_high_prec", "beta": 500, "wmax": 0.18, "iaft_prec": "high"},
    {"name": "low_precision", "beta": 500, "wmax": 1.8, "iaft_prec": "low"},
]

for case in cases:

    gw_params = {
        "restart": False,
        "output": f"gw_{case['name']}",
        "niter": 1,
        "beta": case["beta"],
        "wmax": case["wmax"],
        "iaft_prec": case["iaft_prec"],
    }
    print(f"\n=== Running {case['name']} ===")
    coqui.run_gw(params=gw_params, h_int=thc)

### Section 3: Post-processing - interpolate and visualize GW spectra

<figure style="text-align: center;">

 <img src="../../images/coqui_workflow_pproc.png" alt="Post-processing in CoQuí" width="60%">

 <figcaption><em>Figure 3:</em> Post-processing for spectral analysis.</figcaption>

</figure>

After GW, the checkpoint file `{output}.mbpt.h5` would contain Green's function on a coarse k-mesh and imaginary axis. Post-processing interpolates to high-symmetry paths and analytically continues to the real-axis spectra:

$$
A(k,\omega) = -\frac{1}{\pi}\,\mathrm{Im}\,\mathrm{Tr}\,G(k,\omega+i\eta).
$$

#### 🔹 Spectral interpolation

> ```python
> post_proc.spectral_interpolation(mf: Mf, params: dict)
> ```

Function inputs:
- `mf`: `Mf` object that defines your system (from Section 1).

- `params`: dictionary of interpolation parameters.

Key parameters in `params`:

- `outdir`, `prefix`: location of GW checkpoint data.

- `wannier_file`: MLWF file used for interpolation.

- `ac_alg`, `Nfit`: analytic continuation controls.

- `w_min`, `w_max`, `Nw`, `eta`: real-axis frequency grid.

- `kpath`, `bands_num_npoints`: high-symmetry path sampling.

For complete options, use `help(spectral_interpolation)`.



#### 🔹 Example

In the following example, we interpolate precomputed silicon GW data from a coarse mesh and generate spectral data along a high-symmetry path.

In [ ]:
from coqui.post_proc import spectral_interpolation

qe_dir = "out_777"
wan_h5 = "mlwf_777/si.mlwf.h5"
coqui_dir = "coqui_777"

mf_params = {
    "prefix": "si",
    "outdir": qe_dir,
    "nbnd": 20,
}
mf = coqui.make_mf(coqui_mpi, params=mf_params, mf_type="qe")

winter_params = {
    "outdir": coqui_dir,
    "prefix": "si",
    "wannier_file": wan_h5,
    "ac_alg": "pade",
    "Nfit": 32,
    "w_min": -0.15,
    "w_max": 0.15,
    "Nw": 250,
    "eta": 0.01,
    "bands_num_npoints": 50,
    "kpath": """
      W 0.50 0.25 0.75
      G 0.00 0.00 0.00
      X 0.50 0.00 0.50
    """,

}
spectral_interpolation(mf=mf, params=winter_params)

#### 🔹 Hands-on — Visualize interpolated spectral and quasiparticle bands

>```python
>plot_utils.spectral_plot(
>    ax,
>    "coqui_777/si.mbpt.h5",
>    iteration=0,
>    vmax=10,
>)
>```

function inputs:
- `ax`: matplotlib axis object.
- `mbpt_file`: GW checkpoint file containing spectral data.
- `iteration`: which Dyson-SCF iteration to plot (e.g., `0` for PBE, `1` for $G_0W_0$).
- `vmax`: maximum spectral intensity for color scaling.

Extend the example by plotting $A(k,\omega)$ and overlaying PBE and $G_0W_0$ bands.

Steps:

1. Run interpolation if needed for your chosen path.

2. Plot spectral intensity from `si.mbpt.h5`.

3. Overlay `iteration=0` and `iteration=1` bands from `si_qpg0w0.mbpt.h5`.

4. Adjust `ylim` and `vmax` to highlight the gap region.


Checklist:

- Identify a GW band-gap opening relative to PBE.

- Compare full-frequency spectra with quasiparticle $G_0W_0$ bands.


In [ ]:
import matplotlib.pyplot as plt
import coqui.post_proc.plot_utils as plot_utils

coqui_dir = "coqui_777"
plot_file = f"{coqui_dir}/si.mbpt.h5"
qp_file = f"{coqui_dir}/si_qpg0w0.mbpt.h5"

fig, ax = plt.subplots(1, figsize=(7, 5.5), dpi=100)

# Step 1: Plot spectral function A(k, ω)
plot_utils.spectral_plot(
    ax,
    plot_file,
    calc_type="mbpt",
    vmax=15,
)

# Step 2: Overlay PBE and G0W0 bands
plot_utils.band_plot(
    ax,
    qp_file,
    iteration=0,
    color="tab:red",
    label="PBE",

)

plot_utils.band_plot(
    ax,
    qp_file,
    iteration=1,
    color="tab:blue",
    label="G0W0",
)

ax.axhline(y=0.0, color="black", linestyle="-", linewidth=2.0, alpha=0.5)

ax.set_ylim(-4, 4)
ax.legend(loc=4, fontsize=12)
plt.show()